# Uso de script de modularización

### Paso 1 — Carga del panel de ventanas electorales y Revisión de variables



In [ ]:
import sys
import pandas as pd

general_path = "/workspaces/analisis-politica-economia/"
data_path = f"{general_path}data/tfi_data/"
sys.path.insert(0, f"{general_path}/src")  
import ml_models
from ml_models.cargar_panel import cargar_panel, columnas_candidatas 
import importlib
import ml_models.lasso
from ml_models.lasso import *
importlib.reload(ml_models.lasso)

NIVELES = ["municipal", "provincial", "nacional"]
paneles = {nivel: cargar_panel(nivel,f"{data_path}panel_ventanas.csv") for nivel in NIVELES}

for nivel, df in paneles.items():
    print(f"{nivel}: {df.shape[0]} filas x {df.shape[1]} columnas")

In [ ]:
cols_vc_por_nivel = {}
corr_por_nivel = {}
for nivel in NIVELES: 
    df = paneles[nivel]
    cols_vc = columnas_candidatas(df, excluir_adicional=["delta_participacion_pct"])

    corr = df[cols_vc].corr(method="pearson")  # pairwise, ignora NaN automáticamente
    cols_vc_por_nivel[nivel] = cols_vc
    corr_por_nivel[nivel] = corr
for nivel in NIVELES:
    print(f"{nivel}: {len(cols_vc)} variables candidatas, N={len(df)}")
corr_por_nivel["municipal"] 

### Paso 2 — Sub-selección: colapsar clusters redundantes

La matriz de correlación mostró un cluster de colinealidad casi perfecta entre `ipc_*` y `tc_oficial_*` (r > 0.98 en todo el bloque), además de pares menores en `desocupacion`, `resultado_fiscal` y `salario_real`. Se busca simplificar reduciendo los datos que son redundantes para evitar que LASSO elija de forma aleatoria entre esas opciones.

**Parametrización fijada:**

| Decisión | Valor |
|---|---|
| Umbral de redundancia | `\|r\| ≥ 0.90` |
| Alcance | Transitivo (single-linkage): si A-B≥0.90 y B-C≥0.90, A/B/C van al mismo cluster aunque A-C no llegue al umbral |
| Desempate 1 | Sufijo `_nivel_vc` preferido sobre `_final`/`_pendiente`/`_volatilidad`/`_acum` |
| Desempate 2 | Prioridad teórica de la variable (`ipc` > `desocupacion` > `icg` > `icc` > `salario_real` > `tc_oficial` > `reservas` > `resultado_fiscal`) — orden de relevancia en la literatura de voto económico citada, no un criterio estadístico |

In [ ]:
UMBRAL_REDUNDANCIA = 0.90
PRIORIDAD_TEORICA = ["ipc", "desocupacion", "icg", "icc", "salario_real", "tc_oficial", "reservas", "resultado_fiscal", "emae"]
ORDEN_SUFIJO = ["_nivel_vc", "_final_vc", "_pendiente_vc", "_volatilidad_vc", "_acum_vc"]

clusters_por_nivel = {}
columnas_finales_por_nivel = {}

for nivel in NIVELES:
    df = paneles[nivel]
    cols_vc = cols_vc_por_nivel[nivel]
    corr = corr_por_nivel[nivel]
    print(f"\n\nNivel: {nivel}")
    clusters_por_nivel[nivel] = encontrar_redundantes(corr, UMBRAL_REDUNDANCIA)
    columnas_finales_por_nivel[nivel] = [elegir_representante(cl, df, ORDEN_SUFIJO, PRIORIDAD_TEORICA) if len(cl) > 1 else next(iter(cl)) for cl in clusters_por_nivel[nivel]]

    print(f"De {len(cols_vc)} columnas candidatas, quedan {len(columnas_finales_por_nivel[nivel])} tras colapsar clusters (umbral={UMBRAL_REDUNDANCIA})\n")
    for cl in clusters_por_nivel[nivel]:
        if len(cl) > 1:
            elegido = elegir_representante(cl, df, ORDEN_SUFIJO, PRIORIDAD_TEORICA)
            print(f"cluster ({len(cl)}): {sorted(cl)} -> queda: {elegido}")

In [ ]:
datos_final = {}
for nivel in NIVELES:
    X, y = construir_Xy_final(nivel, columnas_finales_por_nivel[nivel], paneles, target="delta_participacion_pct")
    datos_final[nivel] = (X, y)
    print(f"{nivel}: N={len(y)}, P={X.shape[1]}")

In [ ]:
faltantes = columnas_nan("nacional", "nacional_2013_2015", columnas_finales_por_nivel["nacional"], paneles)
print("Columna(s) que rompen nacional_2013_2015:", faltantes)

In [ ]:
for nivel, id_t in [("municipal", "municipal_2001_2003"), ("provincial", "provincial_2001_2003")]:
    faltantes = columnas_nan(nivel, id_t, columnas_finales_por_nivel[nivel],paneles)
    print(f"{id_t}: NaN en -> {faltantes}")


### Paso 3 — LASSO por coordinate descent (implementación propia)

Formulación: `(1/2n)·‖y - Xβ‖² + α·‖β‖₁` (convención sklearn/glmnet). `X` estandarizada a mano (`ddof=0`), `y` centrada; intercepto = `media(y)`, no se penaliza.

`soft_threshold`: operador proximal de L1, da la selección de variables (coeficiente exactamente en cero si `|z| <= alpha`). `lasso_coordinate_descent`: actualiza una coordenada de `β` a la vez hasta que el cambio máximo entre iteraciones sea `< tol`.

In [ ]:
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    X_std, medias, desvios = estandarizar(X_df)
    y_centrado = y_ser.values - y_ser.mean()
    n = len(y_centrado)

    alpha_prueba = 1.0
    beta_manual = lasso_coordinate_descent(X_std, y_centrado, alpha_prueba)

    kkt = verificar_kkt(X_std, y_centrado, beta_manual, alpha_prueba, n)
    print(f"{nivel} - KKT:", kkt)

    # Chequeo 2: alpha=0 debe coincidir con OLS
    beta_alpha_cero = lasso_coordinate_descent(X_std, y_centrado, alpha=0.0, max_iter=5000)
    beta_ols, *_ = np.linalg.lstsq(X_std, y_centrado, rcond=None)
    print(f"{nivel} - Máxima diferencia vs. OLS (alpha=0):", np.max(np.abs(beta_alpha_cero - beta_ols)))

    residuo_manual = y_centrado - X_std @ beta_alpha_cero
    residuo_ols = y_centrado - X_std @ beta_ols

    print(f"{nivel} - Residuo manual (debería ser ~0 si el sistema es subdeterminado):", np.max(np.abs(residuo_manual)))
    print(f"{nivel} - Residuo OLS (debería ser ~0 también):", np.max(np.abs(residuo_ols)))

### Paso 4 — Grilla de alpha + LOO-CV manual

In [ ]:
resultados_cv = {}
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    resultados_cv[nivel] = lasso_loocv_manual(X_df, y_ser, factor_extension=3.0)
    r = resultados_cv[nivel]
    print(f"{nivel}: alpha_min={r['alpha_min']:.4f}  alpha_1se={r['alpha_1se']:.4f}  (techo grilla={r['alphas'][-1]:.4f})")

In [ ]:

#validacion
resultados_cv = {}
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    print(f"{nivel}: X_df.shape={X_df.shape}, len(y_ser)={len(y_ser)}, 'resultado_fiscal_final_vc' in cols: {'resultado_fiscal_final_vc' in X_df.columns}")
    resultados_cv[nivel] = lasso_loocv_manual(X_df, y_ser, factor_extension=3.0)
    r = resultados_cv[nivel]
    print(f"{nivel}: alpha_min={r['alpha_min']:.4f}  alpha_1se={r['alpha_1se']:.4f}  (techo grilla={r['alphas'][-1]:.4f})")

In [ ]:
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    print(f"--- {nivel} ---")
    print(verificar_saturacion(X_df, y_ser, factores=[1, 3, 10]))
    print()

### Paso 6 — Ajuste final: coeficientes y mejora sobre baseline trivial

`baseline_trivial_loocv`: MSE en LOO de predecir el promedio de los demás puntos -- piso de comparación. `mse_en_alpha`: mismo esquema de LOO que `lasso_loocv_manual`, pero para un alpha puntual (se recalcula, no se guarda en la función de CV).

**Resultado:**

| Nivel | Mejora de `alpha_min` sobre baseline | Mejora de `alpha_1se` |
|---|---|---|
| Municipal | +0.0% | +0.0% |
| Provincial | +0.0% | +0.0% |
| Nacional | +17.31% | +0.0% |

Coeficientes en `alpha_min`: solo `emae_pendiente_vc` (-4.62) en nacional; municipal y provincial no seleccionan ninguna variable. En `alpha_1se`: ninguna variable sobrevive en ningún nivel.

In [ ]:
resumen = []
coeficientes_min, coeficientes_1se = {}, {}

for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    r = resultados_cv[nivel]

    base = baseline_trivial_loocv(y_ser)
    mse_min = mse_en_alpha(X_df, y_ser, r["alpha_min"])
    mse_1se = mse_en_alpha(X_df, y_ser, r["alpha_1se"])

    resumen.append({
        "nivel": nivel,
        "baseline_mse": base,
        "mejora_alpha_min_%": 100 * (1 - mse_min / base),
        "mejora_alpha_1se_%": 100 * (1 - mse_1se / base),
    })

    coeficientes_min[nivel] = ajustar_final(X_df, y_ser, r["alpha_min"])
    coeficientes_1se[nivel] = ajustar_final(X_df, y_ser, r["alpha_1se"])

tabla_resumen = pd.DataFrame(resumen).set_index("nivel")
print(tabla_resumen)

In [ ]:
tabla_coef_min = pd.DataFrame(coeficientes_min)
tabla_coef_min = tabla_coef_min[(tabla_coef_min != 0).any(axis=1)]
print("Coeficientes distintos de cero (alpha_min):")
tabla_coef_min

### Chequeo de estabilidad (leave-one-transition-out) — los tres niveles

**Municipal (alpha=9.618, criterio alpha_1se):** ninguna variable sobrevive en ninguna de las 10 corridas.

**Provincial (alpha=10.139, criterio alpha_1se):** ninguna variable sobrevive en ninguna de las 10 corridas.

**Nacional (alpha=18.164, criterio alpha_1se):** ninguna variable sobrevive en ninguna de las 6 corridas.

In [ ]:
ALPHA_PARA_ESTABILIDAD = {
    "municipal": ("alpha_1se", resultados_cv["municipal"]["alpha_1se"]),
    "provincial": ("alpha_1se", resultados_cv["provincial"]["alpha_1se"]),
    "nacional": ("alpha_1se", resultados_cv["nacional"]["alpha_1se"]),
}

for nivel in NIVELES:
    df = paneles[nivel]
    criterio, alpha = ALPHA_PARA_ESTABILIDAD[nivel]
    X_df, y_ser = datos_final[nivel] 
    resultado = estabilidad_seleccion(nivel, alpha, df, columnas_finales_por_nivel[nivel], "delta_participacion_pct", X_df, y_ser)
    sobrevivientes = resultado.loc[:, (resultado != 0).any(axis=0)]

    print(f"--- {nivel} (alpha={alpha:.3f}, criterio={criterio}) ---")
    if sobrevivientes.empty:
        print("Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.\n")
    else:
        print(sobrevivientes)
        print()